# Notebook 03 - TF-IDF và K-Means phân cụm chủ đề

Notebook này biểu diễn bình luận bằng TF-IDF, chọn số cụm K và phân tích xem cụm K-Means có tương đồng với nhãn aspect thực tế hay không.

## 0. Khởi tạo môi trường

In [1]:
from pathlib import Path
import os
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = Path("results/figures")
CSV_DIR = Path("results/csv")
MODEL_DIR = Path("results/models")
for path in [FIG_DIR, CSV_DIR, MODEL_DIR, Path("datasets/cleaned")]:
    path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

## 3.1 Xây dựng TF-IDF

TF-IDF đo mức quan trọng của từ trong từng bình luận so với toàn bộ corpus. `sublinear_tf=True` dùng log-scaling để giảm ảnh hưởng của từ lặp lại quá nhiều trong một bình luận.

In [2]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from src.clustering import (
    build_tfidf,
    compare_cluster_vs_aspect,
    find_optimal_k,
    get_top_keywords,
    plot_pca_clusters,
    plot_wordclouds,
    run_kmeans,
)

train_clean = pd.read_csv("datasets/cleaned/train_clean.csv", encoding="utf-8-sig")
vectorizer, X_tfidf = build_tfidf(
    train_clean["comment"], max_features=3000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
)
print("Shape ma trận TF-IDF:", X_tfidf.shape)
print(f"Nhận xét: Có {X_tfidf.shape[0]} bình luận và {X_tfidf.shape[1]} đặc trưng từ/ngữ.")

row = X_tfidf[0].toarray().ravel()
terms = vectorizer.get_feature_names_out()
top_idx = row.argsort()[-15:][::-1]
sample_tfidf = pd.DataFrame({"term": terms[top_idx], "tfidf": row[top_idx]})
display(sample_tfidf)

plt.figure(figsize=(8, 5))
sns.barplot(data=sample_tfidf, x="tfidf", y="term", palette="mako")
plt.title("Top TF-IDF của một bình luận mẫu")
plt.tight_layout()
plt.savefig(FIG_DIR / "tfidf_sample_top_terms.png", dpi=150)
plt.show()

Shape ma trận TF-IDF: (7786, 3000)
Nhận xét: Có 7786 bình luận và 3000 đặc trưng từ/ngữ.


,term,tfidf
0,cảm_thấy ok,0.264294
1,thegioididong,0.259732
2,giá_thành,0.253810
3,đẹp loa,0.250336
4,nghe to,0.248716
5,wf,0.244248
6,trâu chụp,0.233415
7,túi_tiền,0.221904
8,loa nghe,0.209751
9,bin,0.202388


## 3.2 Tìm K tối ưu

In [3]:
k_scores = find_optimal_k(X_tfidf, k_range=range(2, 11), save_path=FIG_DIR)
display(k_scores.round(4))

best_row = k_scores.sort_values("silhouette", ascending=False).iloc[0]
best_k = int(best_row["k"])
print(f"K được chọn: {best_k}")
print(f"Lý do: K={best_k} có Silhouette Score cao nhất trong khoảng thử nghiệm ({best_row['silhouette']:.4f}).")
print("Nhận xét: Nếu Elbow không có điểm gãy rõ, Silhouette là tiêu chí định lượng giúp chọn K ổn định hơn cho dữ liệu văn bản thưa.")

,k,inertia,silhouette
0,2,7484.2097,0.0056
1,3,7445.7432,0.0048
2,4,7412.3014,0.0058
3,5,7383.2871,0.0067
4,6,7358.6693,0.0073
5,7,7338.4053,0.0075
6,8,7317.5532,0.0079
7,9,7301.4439,0.0077
8,10,7284.2503,0.0086


K được chọn: 10
Lý do: K=10 có Silhouette Score cao nhất trong khoảng thử nghiệm (0.0086).
Nhận xét: Nếu Elbow không có điểm gãy rõ, Silhouette là tiêu chí định lượng giúp chọn K ổn định hơn cho dữ liệu văn bản thưa.


## 3.3 Chạy K-Means với K tối ưu

In [4]:
kmeans_model, labels, final_silhouette = run_kmeans(X_tfidf, best_k, random_state=42)
train_clean["cluster"] = labels

cluster_counts = train_clean["cluster"].value_counts().sort_index()
display(cluster_counts.to_frame("count"))
print(f"Silhouette Score cuối: {final_silhouette:.4f}")

largest_pct = cluster_counts.max() / cluster_counts.sum() * 100
if largest_pct > 60:
    print(f"Cảnh báo: Cụm lớn nhất chiếm {largest_pct:.2f}% tổng dữ liệu. Điều này thường xảy ra khi TF-IDF tạo không gian thưa.")
else:
    print(f"Nhận xét: Cụm lớn nhất chiếm {largest_pct:.2f}%, chưa vượt ngưỡng lệch 60%.")

plt.figure(figsize=(8, 5))
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette="Set3")
plt.title("Số lượng bình luận mỗi cụm K-Means")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_cluster_size.png", dpi=150)
plt.show()

,count
cluster,
0,348
1,481
2,1792
3,493
4,897
5,1614
6,312
7,872
8,588


Silhouette Score cuối: 0.0086
Nhận xét: Cụm lớn nhất chiếm 23.02%, chưa vượt ngưỡng lệch 60%.


## 3.4 Phân tích top keywords mỗi cụm

In [5]:
top_keywords = get_top_keywords(kmeans_model, vectorizer, n_top=10)
for cluster_id, keywords in top_keywords.items():
    display(Markdown(f"### Cụm {cluster_id}"))
    print("Top 10 keywords:", ", ".join(keywords))
    examples = train_clean.loc[train_clean["cluster"] == cluster_id, "comment"].head(3).tolist()
    print("Ví dụ bình luận thực tế trong cụm:")
    for i, comment in enumerate(examples, start=1):
        print(f"{i}. {comment}")
    print("Nhận xét: Dựa trên keywords và ví dụ, có thể suy luận chủ đề chính của cụm trước khi so sánh với aspect thực tế.\n")

### Cụm 0

Top 10 keywords: chụp hình, hình, chụp, đẹp, hình đẹp, máy, pin, mượt, number, mua
Ví dụ bình luận thực tế trong cụm:
1. điện_thoại đẹp màn_hình rõ nghe gọi ok chụp hình đẹp giá tầm quá ok nhân_viên tgdd thoại_sơn an giang cực_kỳ dễ_thương luôn 🥰
2. máy xịn thứ nhất chơi game mượt chíp snap thứ hai pin trâu xài cả ngày còn number thư ba màn_hình đẹp chụp hình sáng
3. mình cảm_thấy oppo ko mượt quảng_cáo máy mình sử_dụng lỗi ứng_dụng number ngày đổi máy khác chụp hình mặt sưng ko đẹp ko đáng bỏ gần number triệu
Nhận xét: Dựa trên keywords và ví dụ, có thể suy luận chủ đề chính của cụm trước khi so sánh với aspect thực tế.



### Cụm 1

Top 10 keywords: vân tay, vân, tay, nhạy, cảm_biến, cảm_biến vân, tay nhạy, number, máy, mở
Ví dụ bình luận thực tế trong cụm:
1. mình mua máy samsung anumbers number ngày nói_chung máy xài ổn trong tầm giá pin trâu chơi game xài number đến number ngày còn chơi game liên_quân tầm number ngày chụp hình đẹp máy mượt lâu_lâu cảm_biến vân tay nhạy ngón tay hơi ướt xíu máy nhận vân tay luôn ốp lưng xài hơi xót xíu mình làm trong môi_trường chăn_nuôi bụi cám kia
2. máy dùng ok cài khóa mk vân tay khuôn_mặt mở ko vuốt mở đc còn ổn bluetooth bình_thường pin lâu nhạc hay
3. điện_thoại mọi thứ ổn cảm_biến vân tay pin hơi tệ giá
Nhận xét: Dựa trên keywords và ví dụ, có thể suy luận chủ đề chính của cụm trước khi so sánh với aspect thực tế.



### Cụm 2

...

Top 10 keywords: pin, number, sạc, nhanh, ngày, máy, tụt, mua, mới, number ngày
Ví dụ bình luận thực tế trong cụm:
1. pin kém còn miễn chê mua number tình_trạng pin còn number ai giống tôi
2. sao điện_thoại mới sạc nó nóng máy quá trời t_tắt tất_cả mạng ứng dung game cấm ko chạy nền ko sạc dùng điện_thoại nóng quá trời sạc number tiếng mới đầy
3. sạc pin number r k numberh k hao nào s mọi người kêu hao pin nhanh s nhỉ mình thasy pin trâu lun
Nhận xét: Dựa trên keywords và ví dụ, có thể suy luận chủ đề chính của cụm trước khi so sánh với aspect thực tế.



### Cụm 8

Top 10 keywords: tầm giá, tầm, trong tầm, giá, trong, tốt, tốt trong, ổn, máy, pin
Ví dụ bình luận thực tế trong cụm:
1. mọi thứ ngon trong tầm giá lần đầu xài xiaomi cảm_thấy hài_lòng về chất_lượng
2. khá ổn trong tầm giá cam đẹp sạc nhanh màn_hình đẹp âm_thanh hay phù_hợp người thích chơi game tuy_nhiên màn_hình hơi tối ngoài_trời
3. máy tốt trong tầm giá pin trâu màn_hình đẹp to camera đẹp chơi tựa game ở cấu_hình ổn cảm_biến vân tay quang_học trong màn_hình hơi chậm k hỗ_trợ chụp đêm chụp trong đêm hơi xấu còn nv phục_vụ nhiệt_tình chu_đáo
Nhận xét: Dựa trên keywords và ví dụ, có thể suy luận chủ đề chính của cụm trước khi so sánh với aspect thực tế.



### Cụm 9

Top 10 keywords: mọi thứ, thứ, mọi, thứ ok, thứ ổn, ok, number, ổn, pin, mình
Ví dụ bình luận thực tế trong cụm:
1. mọi người cập_nhật phần_mềm nó bớt tốn pin mình thử mọi thứ ok vân tay ko nhạy
2. tất_cả mọi thứ ổn hết màn_hình đẹp máy chạy êm pin sạc nhanh màu mặt lưng đẹp tiết ở phần thiết_kế chi loa ngoài đặt ở bên hông thay đặt ở cạnh tay đè loa sử_dụng í lo chính bản_thân đè quá làm loa hư thêm number nhỏ_xíu xiu mới mua hôm giảm numberđ nay xem tuột xuống đươc giảm hẳn numberđ tức ghê 🥴
3. mọi thứ xài khá ok pin tuột nhanh kinh_khủng xài tuột number chực numberp còn number ý game
Nhận xét: Dựa trên keywords và ví dụ, có thể suy luận chủ đề chính của cụm trước khi so sánh với aspect thực tế.



## 3.5 So sánh cụm K-Means với aspect label thực tế

In [6]:
crosstab, cluster_summary = compare_cluster_vs_aspect(train_clean, "cluster", "main_aspect")
display(crosstab)
display(cluster_summary.assign(match_ratio_pct=lambda x: (x["match_ratio"] * 100).round(2)))

cluster_names = dict(zip(cluster_summary["cluster"], cluster_summary["cluster_name"]))
train_clean["cluster_name"] = train_clean["cluster"].map(cluster_names)
good_clusters = (cluster_summary["match_ratio"] >= 0.5).sum()
print(f"Số cụm khớp tốt (match_ratio >= 50%): {good_clusters}/{len(cluster_summary)}")
print("Nhận xét: Cụm có match_ratio cao cho thấy K-Means tìm được chủ đề gần với aspect thực tế; cụm thấp thường trộn nhiều chủ đề.")

main_aspect,BATTERY,CAMERA,DESIGN,FEATURES,GENERAL,OTHERS,PERFORMANCE,PRICE,SCREEN,SER&ACC,STORAGE
cluster,,,,,,,,,,,
0,8,267,0,4,0,2,3,0,64,0,0
1,3,116,5,278,0,0,3,1,74,1,0
2,227,381,85,285,102,31,271,100,278,27,5
3,98,82,28,69,76,0,51,27,43,19,0
4,212,180,1,151,3,8,224,5,112,1,0
5,128,139,53,576,101,60,325,48,151,33,0
6,2,246,0,3,0,5,7,0,49,0,0
7,551,113,0,122,4,7,7,0,66,2,0
8,117,156,10,81,2,0,69,73,77,0,3


,cluster,cluster_name,match_ratio,total,match_ratio_pct
0,0,CAMERA,0.767241,348,76.72
1,1,FEATURES,0.577963,481,57.80
2,2,CAMERA,0.212612,1792,21.26
3,3,BATTERY,0.198783,493,19.88
4,4,PERFORMANCE,0.249721,897,24.97
5,5,FEATURES,0.356877,1614,35.69
6,6,CAMERA,0.788462,312,78.85
7,7,BATTERY,0.631881,872,63.19
8,8,CAMERA,0.265306,588,26.53
9,9,BATTERY,0.280206,389,28.02


Số cụm khớp tốt (match_ratio >= 50%): 4/10
Nhận xét: Cụm có match_ratio cao cho thấy K-Means tìm được chủ đề gần với aspect thực tế; cụm thấp thường trộn nhiều chủ đề.


## 3.6 Visualize và lưu kết quả

In [7]:
plot_pca_clusters(X_tfidf, labels, cluster_names, FIG_DIR / "pca_kmeans.png")
plot_wordclouds(train_clean, "cluster", "comment", FIG_DIR)

train_clean.to_csv(CSV_DIR / "kmeans_clustered.csv", index=False, encoding="utf-8-sig")
joblib.dump(kmeans_model, MODEL_DIR / "kmeans_model.pkl")
joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.pkl")
cluster_summary.to_csv(CSV_DIR / "kmeans_cluster_summary.csv", index=False, encoding="utf-8-sig")
k_scores.to_csv(CSV_DIR / "kmeans_k_scores.csv", index=False, encoding="utf-8-sig")
print("Đã lưu kết quả phân cụm, model và hình trực quan.")

Đã lưu kết quả phân cụm, model và hình trực quan.
